In [ ]:
# Scenario Statement: Global Streaming Platform OptimizationThe Context: You are a Data Engineer at a global video
# streaming service (like Netflix or YouTube). Every second, your servers generate logs for millions of active video streams.
# These logs contain the "Bandwidth Allocation" (how much data is being sent) for every single user currently watching a video.

# The Problem: Due to a new "Ultra-HD" initiative, the company needs to double the bandwidth allocation for every active stream
#  immediately to improve video quality.

# The Challenge:Massive Data: The number of logs is in the millions, making it impossible for
#   a single computer to process them without crashing or taking hours.

# # Real-Time Speed: The infrastructure costs need to be calculated instantly so server capacity can be adjusted in real-time.
# Mapping the Scenario to the CodeCode StepScenario



from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LabSolution").getOrCreate()

In [ ]:
#LOAD DATASET
df = spark.read.csv("/content/spark_lab_dataset.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

+--------------+-----------+----------------+------+--------+----------------+--------+
|transaction_id|customer_id|            name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+----------------+------+--------+----------------+--------+
|             1|       1057|Christian Forbes| South|12935.47|        Purchase|       0|
|             2|       1003|   Justin Harris| South|  3709.3|      Withdrawal|       0|
|             3|       1004|   Peter Schmitt| South| 3837.45|        Purchase|       1|
|             4|       1049|  Molly Gonzalez|  East| 46638.1|      Withdrawal|       1|
|             5|       1045| Wanda Mcfarland| North|24212.31|        Purchase|       1|
+--------------+-----------+----------------+------+--------+----------------+--------+
only showing top 5 rows
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- amount: d

In [ ]:
#DATA EXPLORATION
print("Total Transactions:", df.count())
df.select("region").distinct().show()
df.groupBy("transaction_type").count().show()

Total Transactions: 500
+------+
|region|
+------+
| South|
|  East|
|  West|
| North|
+------+

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|        Purchase|  162|
|          Refund|  172|
|      Withdrawal|  166|
+----------------+-----+



In [ ]:
#DATA CLEANING
df = df.dropna()
df.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- is_fraud: integer (nullable = true)



In [ ]:
#TRANSFORMATIONS
high_value = df.filter(df.amount > 10000)
fraud_df = df.filter(df.is_fraud == 1)
selected = df.select("customer_id", "amount", "region")

high_value.show(5)
fraud_df.show(5)
selected.show(5)

+--------------+-----------+----------------+------+--------+----------------+--------+
|transaction_id|customer_id|            name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+----------------+------+--------+----------------+--------+
|             1|       1057|Christian Forbes| South|12935.47|        Purchase|       0|
|             4|       1049|  Molly Gonzalez|  East| 46638.1|      Withdrawal|       1|
|             5|       1045| Wanda Mcfarland| North|24212.31|        Purchase|       1|
|             6|       1045|  Jennifer Casey| South|35929.02|      Withdrawal|       1|
|             7|       1054|   Melissa Klein| North|26482.51|        Purchase|       1|
+--------------+-----------+----------------+------+--------+----------------+--------+
only showing top 5 rows
+--------------+-----------+---------------+------+--------+----------------+--------+
|transaction_id|customer_id|           name|region|  amount|transaction_type|is_fraud|
+---------

In [ ]:
#AGGREGATIONS
df.groupBy("region").avg("amount").show()
df.groupBy("region").sum("is_fraud").show()
df.groupBy("transaction_type").count().show()

+------+------------------+
|region|       avg(amount)|
+------+------------------+
| South|24863.821007751936|
|  East|23983.666198347106|
|  West| 25563.42611111112|
| North|23597.939032258077|
+------+------------------+

+------+-------------+
|region|sum(is_fraud)|
+------+-------------+
| South|           57|
|  East|           59|
|  West|           58|
| North|           67|
+------+-------------+

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|        Purchase|  162|
|          Refund|  172|
|      Withdrawal|  166|
+----------------+-----+



In [ ]:
#SPARK SQL
df.createOrReplaceTempView("transactions")

spark.sql("SELECT region, COUNT(*) as total FROM transactions GROUP BY region").show()

spark.sql("SELECT * FROM transactions ORDER BY amount DESC LIMIT 5").show()

spark.sql("SELECT region, SUM(is_fraud) as fraud_count FROM transactions GROUP BY region").show()

+------+-----+
|region|total|
+------+-----+
| South|  129|
|  East|  121|
|  West|  126|
| North|  124|
+------+-----+

+--------------+-----------+-----------------+------+--------+----------------+--------+
|transaction_id|customer_id|             name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+-----------------+------+--------+----------------+--------+
|           427|       1036|      Todd Thomas|  West|49969.04|      Withdrawal|       1|
|           235|       1025|       Mark Smith| South|49825.23|          Refund|       0|
|           311|       1081|          Alan Yu|  West|49818.54|        Purchase|       0|
|           117|       1056|     Thomas Smith| South|49629.78|          Refund|       1|
|           305|       1046|Elizabeth Krueger| North|49602.85|        Purchase|       0|
+--------------+-----------+-----------------+------+--------+----------------+--------+

+------+-----------+
|region|fraud_count|
+------+-----------+
| South|      

In [ ]:
#OPTIMIZATION
df.cache()
df.explain()

== Physical Plan ==
InMemoryTableScan [transaction_id#106, customer_id#107, name#108, region#109, amount#110, transaction_type#111, is_fraud#112]
   +- InMemoryRelation [transaction_id#106, customer_id#107, name#108, region#109, amount#110, transaction_type#111, is_fraud#112], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(1) Filter atleastnnonnulls(7, transaction_id#106, customer_id#107, name#108, region#109, amount#110, transaction_type#111, is_fraud#112)
            +- FileScan csv [transaction_id#106,customer_id#107,name#108,region#109,amount#110,transaction_type#111,is_fraud#112] Batched: false, DataFilters: [atleastnnonnulls(7, transaction_id#106, customer_id#107, name#108, region#109, amount#110, trans..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/spark_lab_dataset.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transaction_id:int,customer_id:int,name:string,region:string,amount:double,transaction_typ...




In [ ]:
# Scenario Statement: Healthcare Data Surge Optimization
# The Context: You are a Data Engineer at a global healthcare analytics company. Every second, hospitals worldwide send patient monitoring logs (heart rate, oxygen levels, blood pressure) into your system.

# The Problem: A new global regulation requires doubling the frequency of patient monitoring data collection to ensure higher accuracy in critical care.

# The Challenge:

# Massive Data: Millions of patient logs per second make it impossible for a single computer to process quickly.

# Real‑Time Speed: Doctors need instant dashboards to adjust treatment protocols in ICUs.

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LabSolution").getOrCreate()

In [ ]:
#LOAD DATASET
df = spark.read.csv("/content/healthcare_data_spark_small.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

+-------------------+----------+-----------+-------+---------+----------+------------+-----------------------+------------------------+-----------+----------------+------------+
|          Timestamp|Patient_ID|Hospital_ID| Region|Device_ID|Heart_Rate|Oxygen_Level|Blood_Pressure_Systolic|Blood_Pressure_Diastolic|Temperature|Respiratory_Rate|Alert_Status|
+-------------------+----------+-----------+-------+---------+----------+------------+-----------------------+------------------------+-----------+----------------+------------+
|2026-04-01 08:00:01|     P1001|       H001|    USA|    DV001|        78|          98|                    120|                      80|       98.6|              16|      Normal|
|2026-04-01 08:00:02|     P1002|       H002|  India|    DV002|        92|          95|                    135|                      88|       99.1|              20|     Warning|
|2026-04-01 08:00:03|     P1003|       H003|     UK|    DV003|       110|          91|                    145|

In [ ]:
#DATA EXPLORATION
print("Total Patient Logs:", df.count())

df.select("Region").distinct().show()

df.groupBy("Alert_Status").count().show()

Total Patient Logs: 20
+-------+
| Region|
+-------+
|Germany|
|  India|
|    USA|
|     UK|
+-------+

+------------+-----+
|Alert_Status|count|
+------------+-----+
|    Critical|    6|
|     Warning|    7|
|      Normal|    7|
+------------+-----+



In [ ]:
#DATA CLEANING
df = df.dropna()
df.printSchema()

root
 |-- Timestamp: timestamp (nullable = true)
 |-- Patient_ID: string (nullable = true)
 |-- Hospital_ID: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Device_ID: string (nullable = true)
 |-- Heart_Rate: integer (nullable = true)
 |-- Oxygen_Level: integer (nullable = true)
 |-- Blood_Pressure_Systolic: integer (nullable = true)
 |-- Blood_Pressure_Diastolic: integer (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Respiratory_Rate: integer (nullable = true)
 |-- Alert_Status: string (nullable = true)



In [ ]:
#TRANSFORMATIONS
critical_patients = df.filter(df.Oxygen_Level < 90)

critical_alerts = df.filter(df.Alert_Status == "Critical")

selected = df.select("Patient_ID", "Heart_Rate", "Region")

critical_patients.show(5)
critical_alerts.show(5)
selected.show(5)

+-------------------+----------+-----------+-------+---------+----------+------------+-----------------------+------------------------+-----------+----------------+------------+
|          Timestamp|Patient_ID|Hospital_ID| Region|Device_ID|Heart_Rate|Oxygen_Level|Blood_Pressure_Systolic|Blood_Pressure_Diastolic|Temperature|Respiratory_Rate|Alert_Status|
+-------------------+----------+-----------+-------+---------+----------+------------+-----------------------+------------------------+-----------+----------------+------------+
|2026-04-01 08:00:06|     P1006|       H005|  India|    DV006|       120|          89|                    150|                     100|      101.2|              28|    Critical|
|2026-04-01 08:00:09|     P1009|       H001|    USA|    DV009|       130|          87|                    160|                     105|      102.1|              30|    Critical|
|2026-04-01 08:00:15|     P1015|       H004|Germany|    DV015|       125|          88|                    155|

In [ ]:
#AGGREGATIONS
df.groupBy("Region").avg("Heart_Rate").show()

df.groupBy("Region").avg("Oxygen_Level").show()

df.groupBy("Alert_Status").count().show()

+-------+-----------------+
| Region|  avg(Heart_Rate)|
+-------+-----------------+
|Germany|            95.25|
|  India|98.57142857142857|
|    USA|             98.2|
|     UK|             93.0|
+-------+-----------------+

+-------+-----------------+
| Region|avg(Oxygen_Level)|
+-------+-----------------+
|Germany|             94.5|
|  India|93.71428571428571|
|    USA|             93.8|
|     UK|            94.75|
+-------+-----------------+

+------------+-----+
|Alert_Status|count|
+------------+-----+
|    Critical|    6|
|     Warning|    7|
|      Normal|    7|
+------------+-----+



In [ ]:
#SPARK SQL
df.createOrReplaceTempView("patients")

spark.sql("SELECT Region, COUNT(*) as total FROM patients GROUP BY Region").show()

spark.sql("SELECT * FROM patients ORDER BY Heart_Rate DESC LIMIT 5").show()

spark.sql("SELECT Region, COUNT(*) as critical_count FROM patients WHERE Alert_Status='Critical' GROUP BY Region").show()

+-------+-----+
| Region|total|
+-------+-----+
|Germany|    4|
|  India|    7|
|    USA|    5|
|     UK|    4|
+-------+-----+

+-------------------+----------+-----------+-------+---------+----------+------------+-----------------------+------------------------+-----------+----------------+------------+
|          Timestamp|Patient_ID|Hospital_ID| Region|Device_ID|Heart_Rate|Oxygen_Level|Blood_Pressure_Systolic|Blood_Pressure_Diastolic|Temperature|Respiratory_Rate|Alert_Status|
+-------------------+----------+-----------+-------+---------+----------+------------+-----------------------+------------------------+-----------+----------------+------------+
|2026-04-01 08:00:18|     P1018|       H001|    USA|    DV018|       132|          86|                    162|                     108|      102.3|              31|    Critical|
|2026-04-01 08:00:09|     P1009|       H001|    USA|    DV009|       130|          87|                    160|                     105|      102.1|            

In [ ]:
#OPTIMIZATION
df.cache()
df.explain()

== Physical Plan ==
InMemoryTableScan [Timestamp#560, Patient_ID#561, Hospital_ID#562, Region#563, Device_ID#564, Heart_Rate#565, Oxygen_Level#566, Blood_Pressure_Systolic#567, Blood_Pressure_Diastolic#568, Temperature#569, Respiratory_Rate#570, Alert_Status#571]
   +- InMemoryRelation [Timestamp#560, Patient_ID#561, Hospital_ID#562, Region#563, Device_ID#564, Heart_Rate#565, Oxygen_Level#566, Blood_Pressure_Systolic#567, Blood_Pressure_Diastolic#568, Temperature#569, Respiratory_Rate#570, Alert_Status#571], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(1) Filter atleastnnonnulls(12, Timestamp#560, Patient_ID#561, Hospital_ID#562, Region#563, Device_ID#564, Heart_Rate#565, Oxygen_Level#566, Blood_Pressure_Systolic#567, Blood_Pressure_Diastolic#568, Temperature#569, Respiratory_Rate#570, Alert_Status#571)
            +- FileScan csv [Timestamp#560,Patient_ID#561,Hospital_ID#562,Region#563,Device_ID#564,Heart_Rate#565,Oxygen_Level#566,Blood_Pressure_Systolic#567,Bloo